In [5]:
import osmium
from osmium import osm, filter
from osmium.filter import TagFilter
import h3

In [6]:
from pathlib import Path

path = Path("data/water/california-260203.osm.pbf/").resolve()

print(path)

C:\Users\ddavi\Projects\scenic-route\processing\src\data\water\california-260203.osm.pbf


In [4]:
class ScenicHandler(osmium.SimpleHandler):
    def __init__(self, resolution=8):
        super().__init__()
        self.resolution = resolution
        self.cells = {}  # h3_cell_id -> dict of counts

    def _get_cell(self, lat, lng):
        cell = h3.latlng_to_cell(lat, lng, self.resolution)
        if cell not in self.cells:
            self.cells[cell] = {
                "water": 0,
                "forest": 0,
                "peak": 0,
                "park": 0,
                "viewpoint": 0,
                "urban": 0,
            }
        return self.cells[cell]

    def node(self, n: osm.Node):
        if not n.location.valid():
            return
        lat, lng = n.location.lat, n.location.lon
        tags = n.tags

        if tags.get("tourism") == "viewpoint":
            self._get_cell(lat, lng)["viewpoint"] += 1
        elif tags.get("natural") == "peak":
            self._get_cell(lat, lng)["peak"] += 1

    def way(self, w: osm.Way):
        # Only handle linear water features here — forests/parks/urban are
        # closed ways that osmium also sends to area(), so we handle them
        # there to avoid double-counting.
        tags = w.tags
        if not (
            tags.get("natural") in ("water", "coastline")
            or tags.get("waterway") in ("river", "stream", "canal", "drain")
        ):
            return

        nodes = [n for n in w.nodes if n.location.valid()]
        if not nodes:
            return
        lat = sum(n.location.lat for n in nodes) / len(nodes)
        lng = sum(n.location.lon for n in nodes) / len(nodes)
        self._get_cell(lat, lng)["water"] += 1

    def area(self, a: osm.Area):
        tags = a.tags

        # Check tags before doing any geometry work
        if tags.get("landuse") == "forest" or tags.get("natural") == "wood":
            feature = "forest"
        elif (
            tags.get("leisure") in ("park", "nature_reserve")
            or tags.get("boundary") == "protected_area"
        ):
            feature = "park"
        elif tags.get("landuse") in ("industrial", "commercial"):
            feature = "urban"
        else:
            return

        try:
            outer = next(a.outer_rings())
            nodes = [n for n in outer if n.location.valid()]
            if not nodes:
                return
            lat = sum(n.location.lat for n in nodes) / len(nodes)
            lng = sum(n.location.lon for n in nodes) / len(nodes)
        except (StopIteration, AttributeError):
            return

        self._get_cell(lat, lng)[feature] += 1

### Execute
Runtime (idx, filters): 4m 27.3s  
Runtime (filters): 3m 53.8s  
Runtime (idx): forever  

In [ ]:
handler = ScenicHandler(resolution=8)
handler.apply_file(
    path,
    locations=True,
    # idx="sparse_file_array",
    filters=[
        TagFilter(
            ("tourism", "viewpoint"),
            ("natural", "peak"),
            ("natural", "water"),
            ("natural", "wood"),
            ("natural", "coastline"),
            ("waterway", "river"),
            ("waterway", "stream"),
            ("waterway", "canal"),
            ("landuse", "forest"),
            ("landuse", "industrial"),
            ("landuse", "commercial"),
            ("leisure", "park"),
            ("leisure", "nature_reserve"),
            ("boundary", "protected_area"),
        )
    ],
)

print(f"Parsed {len(handler.cells)} H3 cells")

Parsed 224796 H3 cells


### Runtime: 

In [ ]:
import json

with open("data/output/scenic_cells.json", "w") as f:
    json.dump(handler.cells, f)

print(f"Saved {len(handler.cells)} H3 cells")

NameError: name 'handler' is not defined

In [6]:
import pandas as pd
import json
import os

print(os.getcwd())

c:\Users\ddavi\Projects\scenic-route\processing\src


In [7]:
with open("data/output/scenic_cells_v1.json", "r") as f:
    cells = json.load(f)

# cells = handler.cells

In [ ]:
# Convert to DataFrame for easy scoring
df = pd.DataFrame.from_dict(cells, orient="index")
df.index.name = "h3_cell"
df.reset_index(inplace=True)

# Scenic score formula
df["diversity"] = (df[["water", "forest", "peak", "park", "viewpoint"]] > 0).sum(axis=1)

df["raw_score"] = (
    3 * df["water"]
    + 2 * df["forest"]
    + 3 * df["peak"]
    + 2 * df["park"]
    + 1 * df["viewpoint"]
    + 2 * df["diversity"]
    - 2 * df["urban"]
)

# Normalize to 0–100
df["score"] = (
    (df["raw_score"] - df["raw_score"].min())
    / (df["raw_score"].max() - df["raw_score"].min())
    * 100
).clip(0, 100)

df_ranked = df.sort_values("score", ascending=False)
print(df_ranked[["h3_cell", "score"]].head(20))

                h3_cell       score
62203   8829abc99dfffff  100.000000
5636    8829a0b431fffff   96.505073
62915   8829abc983fffff   95.264938
62040   8829abc9d5fffff   92.897407
2642    8829aa269bfffff   90.755355
61951   8829abc98bfffff   90.191657
62733   8829aa2459fffff   88.838782
62545   8829abc9b9fffff   87.147689
65316   8829abc989fffff   80.383315
62419   8829aa3621fffff   80.157835
61842   8829abc981fffff   77.677565
63243   8829ab526dfffff   76.324690
62498   8829abc8b5fffff   76.324690
194384  8829b5b043fffff   75.986471
62325   8829abc995fffff   75.310034
64316   8829abc9c7fffff   73.618940
62330   8829ab1861fffff   73.280722
61984   8829abc9c3fffff   72.604284
62373   8829ab1869fffff   72.266065
65015   88298c926bfffff   71.589628


In [27]:
df_ranked.to_json("data/output/scenic_scores.json", orient="records")

In [9]:
print(df_ranked.columns.tolist())

['h3_cell', 'water', 'forest', 'peak', 'park', 'viewpoint', 'urban', 'diversity', 'raw_score', 'score']
